# Notebook 1: Sequence Alignment --- Needleman-Wunsch & Progressive MSA

**Author:** Chidera Ibe (coibe2)

**Project:** Evolutionary Conservation of the PI3K-alpha Cryptic Drug Binding Site

---

This notebook implements the first stage of the bioinformatics pipeline:

1. Fetching PI3K-alpha ortholog and paralog sequences from Ensembl/UniProt
2. Pairwise alignment using our Needleman-Wunsch implementation (affine gap penalties)
3. Computing an all-vs-all distance matrix
4. Progressive multiple sequence alignment (MSA) guided by a phylogenetic tree
5. Verification against established tools (ClustalOmega / Biopython)
6. Paralog alignment of the four human PI3K catalytic isoforms

## Section 1: Setup & Data Fetching

In [ ]:
import sys
import os

# Add project root to Python path so we can import from src/
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.fetchers import (
    fetch_ensembl_orthologs,
    fetch_ensembl_sequences,
    fetch_all_paralogs,
    fetch_human_pik3ca_sequence,
)
from src.utils import load_blosum62, write_fasta, read_fasta, percent_identity
from src.alignment import (
    needleman_wunsch,
    compute_distance_matrix,
    progressive_msa,
    upgma_tree,
)

# Inline plots
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

print("Imports successful.")

### Fetch PI3K-alpha orthologs from Ensembl

We query the Ensembl REST API for homologs of the human *PIK3CA* gene. Results are cached locally in `data/raw/` so subsequent runs do not hit the API.

In [ ]:
# Fetch ortholog metadata from Ensembl
all_orthologs = fetch_ensembl_orthologs(gene_symbol="PIK3CA", species="human")
print(f"Total orthologs returned by Ensembl: {len(all_orthologs)}")

### Filter to one-to-one orthologs with reasonable sequence lengths

We keep only **one-to-one orthologs** (clearest evolutionary correspondence) and later filter by sequence length: the human PI3K-alpha protein is 1068 amino acids, so we retain sequences between 70% and 130% of that length (748--1388 aa) to exclude truncated or misannotated entries.

In [ ]:
HUMAN_PIK3CA_LENGTH = 1068
MIN_LEN = int(0.70 * HUMAN_PIK3CA_LENGTH)  # 748
MAX_LEN = int(1.30 * HUMAN_PIK3CA_LENGTH)  # 1388

# Keep one-to-one orthologs
one2one = [o for o in all_orthologs if o["type"] == "ortholog_one2one"]
print(f"One-to-one orthologs: {len(one2one)}")

In [ ]:
# Fetch protein sequences for the one-to-one orthologs
ortholog_ids = [o["protein_id"] for o in one2one]
ortholog_seqs_raw = fetch_ensembl_sequences(ortholog_ids)
print(f"Sequences fetched: {len(ortholog_seqs_raw)}")

# Filter by length
ortholog_seqs = {
    pid: seq
    for pid, seq in ortholog_seqs_raw.items()
    if MIN_LEN <= len(seq) <= MAX_LEN
}
print(f"Sequences after length filter ({MIN_LEN}-{MAX_LEN} aa): {len(ortholog_seqs)}")

In [ ]:
# Build a lookup from protein_id -> species name for convenience
id_to_species = {o["protein_id"]: o["species"] for o in one2one}

# Summary: list species that passed filtering
species_list = [id_to_species[pid] for pid in ortholog_seqs if pid in id_to_species]
print(f"\nSpecies retained ({len(species_list)}):")
for sp in sorted(species_list):
    print(f"  - {sp}")

### Fetch human PI3K paralogs

The four human PI3K catalytic isoforms (alpha, beta, gamma, delta) will be used for paralog analysis in Section 6.

In [ ]:
paralogs = fetch_all_paralogs()
print("PI3K paralogs fetched:")
for name, seq in paralogs.items():
    print(f"  {name}: {len(seq)} aa")

---

## Section 2: Needleman-Wunsch Pairwise Alignment

We test our from-scratch Needleman-Wunsch implementation (with affine gap penalties) on representative pairs of PI3K-alpha orthologs.

In [ ]:
# Load BLOSUM62 substitution matrix
blosum62 = load_blosum62()
print("BLOSUM62 loaded. Alphabet size:", len(blosum62))

### Demo: Human vs. Mouse PI3K-alpha

Mouse is a close ortholog of human, so we expect high identity and a strong alignment score.

In [ ]:
# Get human PI3K-alpha sequence
human_seq = fetch_human_pik3ca_sequence()
print(f"Human PI3K-alpha length: {len(human_seq)} aa")

# Find mouse ortholog
mouse_pid = None
for pid, species in id_to_species.items():
    if "mus_musculus" in species.lower() and pid in ortholog_seqs:
        mouse_pid = pid
        break

if mouse_pid:
    mouse_seq = ortholog_seqs[mouse_pid]
    print(f"Mouse ortholog ({mouse_pid}): {len(mouse_seq)} aa")
else:
    print("Mouse ortholog not found in filtered set; using first available ortholog.")
    mouse_pid = list(ortholog_seqs.keys())[0]
    mouse_seq = ortholog_seqs[mouse_pid]

In [ ]:
# Needleman-Wunsch alignment: human vs mouse
aln_human, aln_mouse, score_hm = needleman_wunsch(
    human_seq, mouse_seq, blosum62, gap_open=-10, gap_extend=-0.5
)
pct_id_hm = percent_identity(aln_human, aln_mouse)
print(f"Alignment score: {score_hm:.1f}")
print(f"Percent identity: {pct_id_hm:.1%}")
print(f"Alignment length: {len(aln_human)}")

In [ ]:
# Display first 200 characters of the alignment with match indicators
SHOW = 200
match_line = ""
for a, b in zip(aln_human[:SHOW], aln_mouse[:SHOW]):
    if a == b and a != "-":
        match_line += "|"  # identical
    elif a != "-" and b != "-" and blosum62.get(a, {}).get(b, -1) > 0:
        match_line += ":"  # conservative substitution
    else:
        match_line += " "  # mismatch or gap

print("Human : ", aln_human[:SHOW])
print("         ", match_line)
print("Mouse : ", aln_mouse[:SHOW])

### Verification against Biopython PairwiseAligner

As a sanity check, we compare our NW score against Biopython's built-in global aligner using the same parameters.

In [ ]:
from Bio.Align import PairwiseAligner, substitution_matrices

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

bio_score = aligner.score(human_seq, mouse_seq)

print(f"Our NW score:       {score_hm:.1f}")
print(f"Biopython score:    {bio_score:.1f}")
print(f"Difference:         {abs(score_hm - bio_score):.1f}")

if abs(score_hm - bio_score) < 5:
    print("Scores are in close agreement --- implementation verified.")
else:
    print("Note: score difference may be due to edge handling in affine gap model.")

### Human vs. Zebrafish --- a more distant ortholog

In [ ]:
# Find zebrafish ortholog
zebrafish_pid = None
for pid, species in id_to_species.items():
    if "danio_rerio" in species.lower() and pid in ortholog_seqs:
        zebrafish_pid = pid
        break

if zebrafish_pid:
    zf_seq = ortholog_seqs[zebrafish_pid]
    aln_human_zf, aln_zf, score_zf = needleman_wunsch(
        human_seq, zf_seq, blosum62, gap_open=-10, gap_extend=-0.5
    )
    pct_id_zf = percent_identity(aln_human_zf, aln_zf)
    print(f"Human vs Zebrafish ({zebrafish_pid})")
    print(f"  Alignment score:  {score_zf:.1f}")
    print(f"  Percent identity: {pct_id_zf:.1%}")
    print(f"  Alignment length: {len(aln_human_zf)}")
else:
    print("Zebrafish ortholog not found in filtered set.")

### Dot-plot visualization

A dot-plot shows matching residues between two sequences. Diagonal runs indicate conserved regions; off-diagonal features suggest rearrangements or repeats. We use a sliding window to reduce noise.

In [ ]:
def dotplot(seq1, seq2, window=15, threshold=None, title="Dot Plot"):
    """
    Create a simple dot-plot using a sliding window identity score.
    A dot is placed when >= threshold residues match within the window.
    """
    if threshold is None:
        threshold = int(window * 0.6)  # 60% identity within window

    dots_x, dots_y = [], []
    for i in range(len(seq1) - window + 1):
        for j in range(len(seq2) - window + 1):
            matches = sum(
                1 for k in range(window) if seq1[i + k] == seq2[j + k]
            )
            if matches >= threshold:
                dots_x.append(j)
                dots_y.append(i)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(dots_x, dots_y, s=0.3, color="black", alpha=0.7)
    ax.set_xlabel("Sequence 2 position")
    ax.set_ylabel("Sequence 1 position")
    ax.set_title(title)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

# Dot-plot for human vs mouse (use first 500 residues for speed)
REGION = 500
dotplot(
    human_seq[:REGION], mouse_seq[:REGION],
    window=11, threshold=7,
    title=f"Dot Plot: Human vs Mouse PI3K-alpha (first {REGION} residues)"
)

---

## Section 3: Pairwise Distance Matrix

Computing an all-vs-all NW distance matrix for full-length PI3K-alpha sequences (~1068 aa each) is **very slow** with our pure-Python implementation --- $O(n^2 \cdot L^2)$ where $n$ is the number of sequences and $L$ is sequence length.

For practical purposes, we select ~20 representative orthologs spanning vertebrate diversity and compute the pairwise distance matrix on this subset.

In [ ]:
# Define representative species spanning vertebrate phylogeny
REPRESENTATIVE_SPECIES = [
    "homo_sapiens", "pan_troglodytes", "gorilla_gorilla", "macaca_mulatta",
    "mus_musculus", "rattus_norvegicus", "canis_lupus_familiaris",
    "bos_taurus", "sus_scrofa", "equus_caballus",
    "oryctolagus_cuniculus", "monodelphis_domestica",
    "gallus_gallus", "meleagris_gallopavo",
    "anolis_carolinensis", "xenopus_tropicalis",
    "danio_rerio", "oryzias_latipes", "takifugu_rubripes",
    "latimeria_chalumnae",
]

# Map species -> protein_id for orthologs that passed our filters
species_to_pid = {}
for pid in ortholog_seqs:
    sp = id_to_species.get(pid, "").lower()
    species_to_pid[sp] = pid

# Select subset — use underscores in labels (Newick format cannot handle spaces)
subset_seqs = {}
for sp in REPRESENTATIVE_SPECIES:
    if sp in species_to_pid:
        pid = species_to_pid[sp]
        subset_seqs[sp] = ortholog_seqs[pid]

# Also include human (may already be present as homo_sapiens)
if "homo_sapiens" not in subset_seqs:
    subset_seqs["homo_sapiens"] = human_seq

print(f"Selected {len(subset_seqs)} representative species for distance matrix:")
for name in subset_seqs:
    print(f"  {name}: {len(subset_seqs[name])} aa")

In [ ]:
# NOTE: This cell may take a long time to run (potentially 30+ minutes)
# depending on the number of species selected above. Each pairwise NW alignment
# of ~1068 aa sequences takes several seconds in pure Python.
#
# For faster iteration, you can either:
#   1. Reduce subset_seqs to fewer species
#   2. Truncate sequences to a region of interest (e.g., kinase domain)

print(f"Computing {len(subset_seqs)}x{len(subset_seqs)} distance matrix...")
print(f"This involves {len(subset_seqs) * (len(subset_seqs) - 1) // 2} pairwise alignments.")

dist_matrix, dist_labels = compute_distance_matrix(
    subset_seqs, blosum62, gap_open=-10, gap_extend=-0.5
)

print("Distance matrix computed.")
print(f"Shape: {dist_matrix.shape}")
print(f"Min non-zero distance: {dist_matrix[dist_matrix > 0].min():.4f}")
print(f"Max distance: {dist_matrix.max():.4f}")

In [ ]:
# Visualize the distance matrix as a heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    dist_matrix,
    xticklabels=dist_labels,
    yticklabels=dist_labels,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    square=True,
    cbar_kws={"label": "Jukes-Cantor Distance"},
    ax=ax,
)
ax.set_title("Pairwise Distance Matrix --- PI3K-alpha Orthologs")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---

## Section 4: Progressive MSA

A progressive MSA aligns sequences following a guide tree: at each internal node, the two child groups are merged via profile-profile alignment. We use either:

- A **neighbor-joining guide tree** from Notebook 2 (`data/processed/trees/guide_tree.nwk`), or
- A **UPGMA tree** built from our distance matrix as a fallback.

In [ ]:
# Try to load pre-computed NJ guide tree from Notebook 2
guide_tree_path = os.path.join(os.getcwd(), "..", "data", "processed", "trees", "guide_tree.nwk")

if os.path.exists(guide_tree_path):
    with open(guide_tree_path) as f:
        guide_tree_nwk = f.read().strip()
    print("Loaded NJ guide tree from Notebook 2.")
    print(f"Newick (first 200 chars): {guide_tree_nwk[:200]}...")
else:
    print("Guide tree file not found. Building UPGMA tree as fallback...")
    guide_tree_nwk = upgma_tree(dist_matrix, dist_labels)
    print(f"UPGMA tree built with {len(dist_labels)} leaves.")
    print(f"Newick (first 200 chars): {guide_tree_nwk[:200]}...")

In [ ]:
# Run progressive MSA
print(f"Running progressive MSA on {len(subset_seqs)} sequences...")
print("(This may take several minutes for profile-profile alignments of long sequences.)")

msa_result = progressive_msa(
    subset_seqs, guide_tree_nwk, blosum62,
    gap_open=-10, gap_extend=-0.5
)

# Check alignment properties
aln_lengths = set(len(s) for s in msa_result.values())
print(f"\nMSA complete.")
print(f"Number of sequences: {len(msa_result)}")
print(f"Alignment length(s): {aln_lengths}")
assert len(aln_lengths) == 1, "All aligned sequences must have the same length!"

### Display a slice of the MSA

We examine the alignment around positions corresponding to the DFG motif region (~residues 930-940 in the human sequence). This is a highly conserved kinase activation segment.

In [ ]:
# Display MSA slice around a conserved region
# The DFG motif is near position 930-940 in the ungapped human sequence.
# In the MSA, gap insertions shift column indices, so we estimate.

SLICE_START = 930
SLICE_END = 980

print(f"MSA slice (columns {SLICE_START}-{SLICE_END}):")
print("=" * 70)
for name, seq in msa_result.items():
    aln_len = len(seq)
    # Clamp to alignment length
    start = min(SLICE_START, aln_len)
    end = min(SLICE_END, aln_len)
    print(f"{name:>22s}  {seq[start:end]}")
print("=" * 70)

In [ ]:
# Save the MSA to FASTA
msa_output_path = os.path.join(
    os.getcwd(), "..", "data", "processed", "alignments", "pik3ca_orthologs_msa.fasta"
)
write_fasta(msa_result, msa_output_path)
print(f"MSA saved to: {msa_output_path}")

---

## Section 5: Verification vs ClustalOmega

A full ClustalOmega comparison requires either a local installation (`clustalo` binary) or the EBI REST API. Here we evaluate our MSA quality using the **sum-of-pairs (SP) score**, a standard metric for MSA quality.

The SP score sums all pairwise column scores across the alignment using the BLOSUM62 matrix, rewarding conserved columns and penalizing mismatches and gaps.

In [ ]:
def sum_of_pairs_score(msa, sub_matrix, gap_penalty=-4):
    """
    Compute the sum-of-pairs score for an MSA.

    For each column, score every pair of residues using the substitution matrix.
    Gaps paired with residues receive gap_penalty; gap-gap pairs score 0.

    Parameters
    ----------
    msa : dict
        name -> aligned sequence.
    sub_matrix : dict of dict
        Substitution scores (e.g., BLOSUM62).
    gap_penalty : float
        Score for aligning a residue with a gap.

    Returns
    -------
    total_score : float
    per_column_scores : list of float
    """
    seqs = list(msa.values())
    n_seqs = len(seqs)
    aln_len = len(seqs[0])

    per_column_scores = []
    total_score = 0.0

    for col in range(aln_len):
        col_score = 0.0
        for i in range(n_seqs):
            for j in range(i + 1, n_seqs):
                a = seqs[i][col]
                b = seqs[j][col]
                if a == "-" and b == "-":
                    continue  # gap-gap: no contribution
                elif a == "-" or b == "-":
                    col_score += gap_penalty
                else:
                    col_score += sub_matrix.get(a, {}).get(b, -1)
        per_column_scores.append(col_score)
        total_score += col_score

    return total_score, per_column_scores


print("Computing sum-of-pairs score for our MSA...")
sp_total, sp_per_col = sum_of_pairs_score(msa_result, blosum62)
aln_length = len(list(msa_result.values())[0])
n_seqs = len(msa_result)
n_pairs = n_seqs * (n_seqs - 1) // 2

print(f"\nMSA Quality Metrics")
print(f"{'='*40}")
print(f"Number of sequences:     {n_seqs}")
print(f"Alignment length:        {aln_length}")
print(f"Number of pairs:         {n_pairs}")
print(f"Total SP score:          {sp_total:.1f}")
print(f"Mean SP score per column: {np.mean(sp_per_col):.2f}")
print(f"Median SP score per col:  {np.median(sp_per_col):.2f}")

In [ ]:
# Plot per-column SP scores to see which regions are well-conserved
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sp_per_col, linewidth=0.5, color="steelblue", alpha=0.7)
ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.5)
ax.set_xlabel("Alignment column")
ax.set_ylabel("Sum-of-pairs score")
ax.set_title("Per-Column Sum-of-Pairs Score across the MSA")
plt.tight_layout()
plt.show()

In [ ]:
# Additional quality metrics

# Fraction of gap-free columns
seqs_list = list(msa_result.values())
gap_free_cols = sum(
    1 for col in range(aln_length)
    if all(seqs_list[i][col] != "-" for i in range(n_seqs))
)
print(f"Gap-free columns: {gap_free_cols} / {aln_length} ({gap_free_cols/aln_length:.1%})")

# Fraction of columns with > 80% occupancy (non-gap)
high_occ_cols = sum(
    1 for col in range(aln_length)
    if sum(1 for i in range(n_seqs) if seqs_list[i][col] != "-") / n_seqs >= 0.8
)
print(f"Columns with >= 80% occupancy: {high_occ_cols} / {aln_length} ({high_occ_cols/aln_length:.1%})")

# Average pairwise identity across MSA
identities = []
names = list(msa_result.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pi = percent_identity(msa_result[names[i]], msa_result[names[j]])
        identities.append(pi)
print(f"Average pairwise identity: {np.mean(identities):.1%}")
print(f"Min pairwise identity:     {np.min(identities):.1%}")
print(f"Max pairwise identity:     {np.max(identities):.1%}")

---

## Section 6: Paralog Alignment

The four human PI3K catalytic subunit isoforms (alpha/PIK3CA, beta/PIK3CB, gamma/PIK3CG, delta/PIK3CD) share a common domain architecture but have diverged in sequence. Aligning them reveals which regions are conserved across the family versus isoform-specific.

In [ ]:
# Build a small UPGMA guide tree for the 4 paralogs
print("Computing paralog pairwise distances...")
para_dist, para_labels = compute_distance_matrix(
    paralogs, blosum62, gap_open=-10, gap_extend=-0.5
)

print("\nParalog distance matrix:")
for i, lab in enumerate(para_labels):
    row = "  ".join(f"{para_dist[i, j]:.3f}" for j in range(len(para_labels)))
    print(f"  {lab:>20s}  {row}")

para_tree_nwk = upgma_tree(para_dist, para_labels)
print(f"\nUPGMA guide tree: {para_tree_nwk}")

In [ ]:
# Progressive MSA of the 4 paralogs
print("Running progressive MSA on PI3K paralogs...")
para_msa = progressive_msa(
    paralogs, para_tree_nwk, blosum62,
    gap_open=-10, gap_extend=-0.5
)

para_aln_len = len(list(para_msa.values())[0])
print(f"Paralog MSA complete. Alignment length: {para_aln_len}")

In [ ]:
# Save paralog MSA
para_msa_path = os.path.join(
    os.getcwd(), "..", "data", "processed", "alignments", "pik3_paralogs_msa.fasta"
)
write_fasta(para_msa, para_msa_path)
print(f"Paralog MSA saved to: {para_msa_path}")

In [ ]:
# Show alignment slices at regions of interest

# Region 1: N-terminal region (adaptor-binding domain)
print("Paralog MSA --- N-terminal region (columns 0-80):")
print("=" * 90)
for name, seq in para_msa.items():
    print(f"{name:>20s}  {seq[:80]}")
print()

# Region 2: Near the kinase domain / DFG motif region
mid = para_aln_len // 2
print(f"Paralog MSA --- Mid-region (columns {mid}-{mid+80}):")
print("=" * 90)
for name, seq in para_msa.items():
    print(f"{name:>20s}  {seq[mid:mid+80]}")
print()

# Region 3: C-terminal kinase domain
ct_start = max(0, para_aln_len - 100)
print(f"Paralog MSA --- C-terminal region (columns {ct_start}-{para_aln_len}):")
print("=" * 90)
for name, seq in para_msa.items():
    print(f"{name:>20s}  {seq[ct_start:]}")

In [ ]:
# Paralog pairwise identity summary
print("Pairwise identity between PI3K paralogs:")
para_names = list(para_msa.keys())
for i in range(len(para_names)):
    for j in range(i + 1, len(para_names)):
        pi = percent_identity(para_msa[para_names[i]], para_msa[para_names[j]])
        print(f"  {para_names[i]:>20s} vs {para_names[j]:<20s}: {pi:.1%}")

---

## Summary

In this notebook, we:

1. **Fetched** PI3K-alpha ortholog sequences from Ensembl and paralog sequences from UniProt
2. **Aligned** representative pairs using our Needleman-Wunsch implementation with affine gap penalties, and verified against Biopython
3. **Computed** an all-vs-all pairwise distance matrix (Jukes-Cantor corrected) for ~20 representative vertebrate orthologs
4. **Built** a progressive MSA guided by either a neighbor-joining tree (from Notebook 2) or a UPGMA fallback tree
5. **Evaluated** MSA quality using sum-of-pairs scoring and alignment statistics
6. **Aligned** the four human PI3K catalytic isoforms to compare paralog divergence

**Next:** Notebook 2 will use the distance matrix to build phylogenetic trees (neighbor-joining and maximum parsimony) for evolutionary analysis.